# Тема 5. Ансамблевые методы и случайный лес

На прошлом занятии мы научились строить и диагностировать модели. Теперь сделаем следующий шаг: вместо одной модели будем обучать **много моделей** и объединять их предсказания. Такой подход называется **ансамблевым**.

Ансамблевые методы, как правило, превосходят любую одиночную модель. Именно они доминируют в соревнованиях по ML на Kaggle и часто используются в промышленных системах.

### Содержание
1. [Зачем нужны ансамбли](#1.-Зачем-нужны-ансамбли)
2. [Бутстрап и бэггинг](#2.-Бутстрап-и-бэггинг)
3. [Out-of-bag ошибка](#3.-Out-of-bag-ошибка)
4. [Случайный лес: алгоритм](#4.-Случайный-лес:-алгоритм)
5. [Параметры и настройка](#5.-Параметры-и-настройка)
6. [Разброс и декорреляция деревьев](#6.-Разброс-и-декорреляция-деревьев)
7. [Важность признаков](#7.-Важность-признаков)
8. [Плюсы и минусы случайного леса](#8.-Плюсы-и-минусы-случайного-леса)
9. [Практика](#9.-Практика)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
sns.set()
from matplotlib import pyplot as plt

from sklearn.datasets import load_iris, make_circles
from sklearn.ensemble import (
    BaggingClassifier, RandomForestClassifier, RandomForestRegressor, ExtraTreesClassifier
)
from sklearn.model_selection import (
    GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
)
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score

%config InlineBackend.figure_format = 'svg'

---
## 1. Зачем нужны ансамбли

### Теорема Кондорсе (1784)

Представьте жюри из $N$ независимых членов, каждый из которых принимает правильное решение с вероятностью $p > 0.5$. Какова вероятность, что **большинство** проголосует правильно?

Теорема Кондорсе о жюри присяжных гласит: эта вероятность **растёт с $N$ и стремится к 1**.

Простой пример: $p = 0.7$, $N = 3$ независимых классификатора. Ансамбль (мажоритарное голосование) ошибётся только если ошиблись двое или трое:

$$P(\text{ошибка}) = \binom{3}{2}(0.3)^2(0.7) + (0.3)^3 = 0.189$$

То есть ансамбль трёх моделей с точностью 70% даёт **81.1%** — лучше каждой по отдельности.

> Оговорка: теорема работает только при **независимости** ошибок. Если все модели делают одинаковые ошибки, ансамбль ничего не даст. Именно поэтому так важно, чтобы модели в ансамбле были **разнообразными**.

In [ ]:
# Покажем эффект теоремы Кондорсе
from scipy.stats import binom

def ensemble_accuracy(p_single, n_models):
    """Точность мажоритарного голосования n_models независимых классификаторов."""
    majority = n_models // 2 + 1
    return sum(binom.pmf(k, n_models, p_single) for k in range(majority, n_models + 1))

p_values = np.linspace(0.5, 1.0, 100)
fig, ax = plt.subplots(figsize=(8, 4))

for n in [1, 3, 7, 21, 101]:
    accs = [ensemble_accuracy(p, n) for p in p_values]
    ax.plot(p_values, accs, label=f'n={n}')

ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Точность одной модели $p$')
ax.set_ylabel('Точность ансамбля')
ax.set_title('Теорема Кондорсе: эффект от числа независимых классификаторов')
ax.legend()
ax.grid(True, alpha=0.3)

При $p < 0.5$ ансамбль только ухудшает ситуацию — кривые уходят вниз. При $p > 0.5$ чем больше моделей, тем лучше результат.

### Математика: почему ансамбль снижает ошибку

Рассмотрим задачу регрессии. Есть $n$ базовых алгоритмов $b_1(x), \dots, b_n(x)$, каждый с ошибкой $\varepsilon_i(x) = b_i(x) - y(x)$.

Ожидаемый квадрат ошибки одного алгоритма:
$$\mathbb{E}_x\left[\varepsilon_i^2(x)\right] = E$$

Ожидаемый квадрат ошибки их среднего $\frac{1}{n}\sum_i b_i(x)$:
$$\mathbb{E}_x\left[\left(\frac{1}{n}\sum_i \varepsilon_i(x)\right)^2\right]$$

**Если ошибки некоррелированы** ($\mathbb{E}[\varepsilon_i \varepsilon_j] = 0$ при $i \neq j$):
$$= \frac{1}{n^2} \sum_i \mathbb{E}[\varepsilon_i^2] = \frac{E}{n}$$

**Вывод:** среднее $n$ некоррелированных алгоритмов снижает ошибку в $n$ раз. Главная задача при построении ансамбля — сделать модели как можно более **разнообразными**.

---
## 2. Бутстрап и бэггинг

### Бутстрап

**Бутстрап** — статистический приём генерации новых выборок из имеющихся данных.

Алгоритм прост: из исходной выборки размером $\ell$ берём $\ell$ объектов **с возвращением**. Некоторые объекты попадут несколько раз, некоторые — не попадут вовсе. В среднем каждый объект **не попадает** в выборку с вероятностью:

$$\left(1 - \frac{1}{\ell}\right)^\ell \xrightarrow{\ell \to \infty} e^{-1} \approx 0.368$$

То есть примерно **37% объектов** оказываются «за бортом» каждой бутстраповской выборки — эти объекты называются **out-of-bag (OOB)**.

Посмотрим на практике: используем датасет телеком-оператора и оценим среднее количество звонков в службу поддержки для лояльных и ушедших клиентов.

In [ ]:
df = pd.read_csv("../../data/telecom_churn.csv")

fig, ax = plt.subplots(figsize=(8, 4))
df.loc[df["Churn"] == False, "Customer service calls"].hist(
    ax=ax, alpha=0.7, label="Лояльные", color='steelblue'
)
df.loc[df["Churn"] == True, "Customer service calls"].hist(
    ax=ax, alpha=0.7, label="Ушедшие", color='orange'
)
ax.set_xlabel("Количество звонков в поддержку")
ax.set_ylabel("Частота")
ax.set_title("Распределение звонков в поддержку")
ax.legend()
ax.grid(True, alpha=0.3)

In [ ]:
def bootstrap_ci(data, n_samples=1000, alpha=0.05, statistic=np.mean):
    """Доверительный интервал для статистики методом бутстрапа."""
    bootstrapped = np.array([
        statistic(np.random.choice(data, size=len(data), replace=True))
        for _ in range(n_samples)
    ])
    lower = np.percentile(bootstrapped, 100 * alpha / 2)
    upper = np.percentile(bootstrapped, 100 * (1 - alpha / 2))
    return lower, upper, bootstrapped

np.random.seed(42)

loyal_calls   = df.loc[df["Churn"] == False, "Customer service calls"].values
churn_calls   = df.loc[df["Churn"] == True,  "Customer service calls"].values

lo_l, hi_l, bs_l = bootstrap_ci(loyal_calls)
lo_c, hi_c, bs_c = bootstrap_ci(churn_calls)

print(f"Лояльные  — среднее: {loyal_calls.mean():.2f}, 95% CI: [{lo_l:.2f}, {hi_l:.2f}]")
print(f"Ушедшие   — среднее: {churn_calls.mean():.2f}, 95% CI: [{lo_c:.2f}, {hi_c:.2f}]")

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, bs, title, color in zip(
    axes, [bs_l, bs_c],
    ['Лояльные: среднее кол-во звонков', 'Ушедшие: среднее кол-во звонков'],
    ['steelblue', 'orange']
):
    ax.hist(bs, bins=30, color=color, alpha=0.7, edgecolor='white')
    ax.axvline(bs.mean(), color='black', linewidth=2, label='Среднее')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()

Доверительные интервалы не пересекаются — разница в количестве звонков между группами статистически значима.

### Бэггинг (Bootstrap Aggregating)

**Бэггинг** был предложен Лео Брейманом в 1994 году. Идея проста:

1. Сгенерировать $M$ бутстраповских выборок $X_1, \dots, X_M$ из обучающего множества.
2. На каждой выборке обучить отдельную модель $a_i(x)$.
3. Объединить предсказания:
   - **Регрессия:** среднее: $a(x) = \frac{1}{M}\sum_{i=1}^M a_i(x)$
   - **Классификация:** голосование: $a(x) = \text{mode}\{a_1(x), \dots, a_M(x)\}$

Модели разнообразны, потому что обучались на разных выборках. Ошибки частично не коррелированы → ансамбль точнее.

Посмотрим на бэггинг против одного дерева на синтетических данных:

In [ ]:
np.random.seed(42)

def f_true(x):
    return np.exp(-x**2) + 1.5 * np.exp(-(x - 2)**2)

def generate_data(n, noise=0.1):
    X = np.sort(np.random.rand(n) * 10 - 5)
    y = f_true(X) + np.random.normal(0, noise, n)
    return X.reshape(-1, 1), y

X_train, y_train = generate_data(150)
X_test,  y_test  = generate_data(1000)
x_line = np.linspace(-5, 5, 300).reshape(-1, 1)

# Одиночное дерево
tree_single = DecisionTreeClassifier.__bases__[0]  # workaround — используем напрямую
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor

tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)

bagging = BaggingRegressor(estimator=DecisionTreeRegressor(), n_estimators=100, random_state=42)
bagging.fit(X_train, y_train)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, model, title in zip(
    axes,
    [tree, bagging],
    ['Одиночное дерево решений', 'Бэггинг (100 деревьев)']
):
    ax.scatter(X_train, y_train, s=15, alpha=0.5, color='gray', label='Обучение')
    ax.plot(x_line, f_true(x_line.ravel()), 'k--', lw=2, label='Истинная функция')
    ax.plot(x_line, model.predict(x_line), color='steelblue', lw=2, label='Предсказание')
    mse = np.mean((model.predict(X_test) - y_test)**2)
    ax.set_title(f'{title}\nMSE = {mse:.4f}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
plt.tight_layout()

Бэггинг даёт значительно более гладкую и точную аппроксимацию — ошибки отдельных деревьев усредняются.

Посмотрим то же самое в задаче классификации на данных из двух кругов:

In [ ]:
np.random.seed(42)
X_c, y_c = make_circles(n_samples=500, factor=0.1, noise=0.35, random_state=42)
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

models = {
    'Дерево решений': DecisionTreeClassifier(random_state=42),
    'Бэггинг (100 деревьев)': BaggingClassifier(
        estimator=DecisionTreeClassifier(), n_estimators=100, random_state=42, n_jobs=-1
    ),
}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
x_range = np.linspace(X_c.min(), X_c.max(), 150)
xx1, xx2 = np.meshgrid(x_range, x_range)

for ax, (title, model) in zip(axes, models.items()):
    model.fit(X_tr_c, y_tr_c)
    Z = model.predict(np.c_[xx1.ravel(), xx2.ravel()]).reshape(xx1.shape)
    ax.contourf(xx1, xx2, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X_c[:, 0], X_c[:, 1], c=y_c, cmap='coolwarm', s=20, edgecolors='white')
    acc = accuracy_score(y_te_c, model.predict(X_te_c))
    ax.set_title(f'{title}\nТочность: {acc:.3f}')
    ax.grid(True, alpha=0.2)
plt.tight_layout()

Граница решения бэггинга значительно **сглаженнее** — меньше острых углов, меньше переобучения.

---
## 3. Out-of-bag ошибка

Помните, что ~37% объектов не попадают в каждую бутстраповскую выборку? Их можно использовать для **оценки качества без кросс-валидации**.

**Алгоритм:**
1. Для каждого объекта $x_i$ собираем предсказания только тех деревьев, в обучение которых $x_i$ **не входил**.
2. Финальное предсказание — голосование / среднее по этим деревьям.
3. OOB-ошибка = ошибка на этих предсказаниях.

Преимущество: не нужно тратить время на отдельную кросс-валидацию. OOB-оценка получается «бесплатно» в процессе обучения.

В sklearn включается параметром `oob_score=True`:

In [ ]:
# Загружаем данные о телеком-клиентах
df_churn = pd.read_csv("../../data/telecom_churn.csv")

# Отбираем числовые признаки
num_cols = [c for c in df_churn.columns if df_churn[c].dtype in ['float64', 'int64']]
X_ch = df_churn[num_cols].copy()
y_ch = (df_churn["Churn"]).astype(int).values

rf_oob = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42, n_jobs=-1)
rf_oob.fit(X_ch, y_ch)

print(f"OOB-точность:            {rf_oob.oob_score_:.4f}")
print(f"CV-точность (5-fold):    {cross_val_score(rf_oob, X_ch, y_ch, cv=5).mean():.4f}")
print("\nОценки близки — OOB работает как полноценная замена кросс-валидации.")

---
## 4. Случайный лес: алгоритм

Случайный лес — это улучшенный бэггинг над деревьями решений. Лео Брейман и Адель Катлер развили идею, добавив **случайность при выборе признаков**.

### Алгоритм построения леса из $N$ деревьев

Для каждого дерева $k = 1, \dots, N$:
1. Сгенерировать бутстраповскую выборку $X_k$.
2. Строить дерево рекурсивно:
   - В каждом узле выбрать **случайное подмножество** из $m$ признаков (из $d$ доступных).
   - Найти лучшее разбиение **только среди этих $m$ признаков**.
   - Обычно $m = \sqrt{d}$ для классификации, $m = d/3$ для регрессии.
3. Строить дерево до полного роста (без обрезки).

**Ключевое отличие от бэггинга:** в бэггинге на каждом шаге рассматриваются **все** признаки; в случайном лесе — только случайное подмножество. Это **декоррелирует** деревья, снижая дисперсию ансамбля.

In [ ]:
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor

np.random.seed(42)
X_train_r, y_train_r = generate_data(150)
X_test_r,  y_test_r  = generate_data(1000)

models_reg = {
    'Одиночное дерево': DecisionTreeRegressor(random_state=42),
    'Бэггинг (10 деревьев)': BaggingRegressor(
        estimator=DecisionTreeRegressor(), n_estimators=10, random_state=42
    ),
    'Случайный лес (10 деревьев)': RandomForestRegressor(
        n_estimators=10, random_state=42
    ),
}

results = {}
for name, model in models_reg.items():
    model.fit(X_train_r, y_train_r)
    mse = np.mean((model.predict(X_test_r) - y_test_r)**2)
    results[name] = mse

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, model) in zip(axes, models_reg.items()):
    ax.scatter(X_train_r, y_train_r, s=10, alpha=0.4, color='gray')
    ax.plot(x_line, f_true(x_line.ravel()), 'k--', lw=2, label='Истинная')
    ax.plot(x_line, model.predict(x_line), color='steelblue', lw=2, label='Предсказание')
    ax.set_title(f'{name}\nMSE = {results[name]:.4f}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
plt.tight_layout()

---
## 5. Параметры и настройка

Основные гиперпараметры `RandomForestClassifier` в sklearn:

| Параметр | Что регулирует | Типичные значения |
|---|---|---|
| `n_estimators` | Число деревьев | 100–500 (больше = лучше, но дороже) |
| `max_depth` | Макс. глубина дерева | `None` (полный рост) или 5–30 |
| `min_samples_leaf` | Мин. объектов в листе | 1–20 |
| `max_features` | Признаков при каждом сплите | `'sqrt'` для классификации |
| `oob_score` | Считать OOB-ошибку | `True` |
| `n_jobs` | Параллелизм | `-1` (все ядра) |

Посмотрим, как качество меняется с каждым параметром на задаче предсказания оттока клиентов.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def plot_param_curve(param_grid, param_name, xlabel, fixed_params=None):
    """Строит кривые обучения и CV по одному параметру случайного леса."""
    if fixed_params is None:
        fixed_params = {}
    train_scores, test_scores = [], []
    for val in param_grid:
        params = {'n_estimators': 100, 'random_state': 42, 'n_jobs': -1}
        params.update(fixed_params)
        params[param_name] = val
        rfc = RandomForestClassifier(**params)
        tr, te = [], []
        for tr_idx, te_idx in skf.split(X_ch, y_ch):
            rfc.fit(X_ch.iloc[tr_idx], y_ch[tr_idx])
            tr.append(accuracy_score(y_ch[tr_idx], rfc.predict(X_ch.iloc[tr_idx])))
            te.append(accuracy_score(y_ch[te_idx], rfc.predict(X_ch.iloc[te_idx])))
        train_scores.append(tr)
        test_scores.append(te)
    train_scores = np.array(train_scores)
    test_scores  = np.array(test_scores)
    mu_tr, std_tr = train_scores.mean(1), train_scores.std(1)
    mu_te, std_te = test_scores.mean(1),  test_scores.std(1)
    plt.plot(param_grid, mu_tr, color='steelblue', alpha=0.8, label='Обучение')
    plt.plot(param_grid, mu_te, color='orange',    alpha=0.8, label='Валидация (CV)')
    plt.fill_between(param_grid, mu_te - std_te, mu_te + std_te, color='orange', alpha=0.2)
    plt.fill_between(param_grid, mu_te - 2*std_te, mu_te + 2*std_te, color='orange', alpha=0.1)
    plt.xlabel(xlabel)
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True, alpha=0.3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

plt.sca(axes[0])
plot_param_curve([5, 10, 15, 20, 30, 50, 75, 100], 'n_estimators', 'Число деревьев')
axes[0].set_title('n_estimators')

plt.sca(axes[1])
plot_param_curve([3, 5, 7, 9, 11, 13, 15, 17, 20], 'max_depth', 'Максимальная глубина',
                 fixed_params={'n_estimators': 100})
axes[1].set_title('max_depth')

plt.sca(axes[2])
plot_param_curve([1, 3, 5, 7, 9, 11, 13, 15], 'min_samples_leaf', 'Мин. объектов в листе',
                 fixed_params={'n_estimators': 100})
axes[2].set_title('min_samples_leaf')

plt.tight_layout()
plt.suptitle('Влияние гиперпараметров на качество случайного леса', y=1.02, fontsize=13)

**Выводы:**
- **`n_estimators`**: качество растёт и выходит на плато. Больше деревьев = лучше, но дороже. 100–200 деревьев обычно достаточно.
- **`max_depth`**: регуляризует модель. Без ограничения — 100% точность на обучении (переобучение). Оптимум: 10–15 для этого датасета.
- **`min_samples_leaf`**: тоже регуляризует. Уменьшает переобучение, незначительно снижая качество.

### GridSearchCV для подбора оптимальных параметров

In [ ]:
parameters = {
    'max_features':      [4, 7, 10, 13],
    'min_samples_leaf':  [1, 3, 5, 7],
    'max_depth':         [5, 10, 15, 20],
}
rfc = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
gcv = GridSearchCV(rfc, parameters, n_jobs=-1, cv=skf, scoring='accuracy')
gcv.fit(X_ch, y_ch)

print("Лучшие параметры:", gcv.best_params_)
print(f"Лучшая CV-точность:  {gcv.best_score_:.4f}")

---
## 6. Разброс и декорреляция деревьев

Почему случайный лес работает лучше бэггинга?

Дисперсию случайного леса можно записать как:

$$\text{Var}\, f_{RF}(x) = \rho(x) \cdot \sigma^2(x)$$

где:
- $\sigma^2(x)$ — дисперсия одного дерева (шум),
- $\rho(x)$ — **корреляция** между любыми двумя деревьями.

**Случайный выбор признаков уменьшает $\rho$** — деревья становятся разнообразнее, и ансамбль точнее.

### ExtraTrees: ещё больше случайности

`ExtraTreesClassifier` идёт дальше: пороги для разбиения выбираются **случайно**, а не оптимально. Это ещё сильнее декоррелирует деревья, снижая дисперсию ценой небольшого роста смещения. На многих задачах работает не хуже или даже лучше случайного леса.

In [ ]:
models_compare = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Extra Trees':   ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}
for name, model in models_compare.items():
    score = cross_val_score(model, X_ch, y_ch, cv=5, scoring='roc_auc').mean()
    print(f"{name:20s}: ROC-AUC = {score:.4f}")

---
## 7. Важность признаков

Одно из главных преимуществ случайного леса — встроенная **оценка важности признаков**. Это инструмент интерпретации: понять, на что модель опирается при предсказании.

### Важность на основе снижения примеси (Gini Importance)

Идея: чем больше снижение неоднородности (Джини или MSE) даёт признак $X_j$ при разбиении узла, тем он важнее.

Формально, важность признака $X_j$ в дереве $t$:

$$\text{VI}^{(t)}(X_j) = \frac{1}{N_t}\sum_{\text{узел } v: X_j \text{ использован}} \Delta I(v)$$

где $\Delta I(v)$ — взвешенное снижение примеси в узле $v$. Итоговая важность — среднее по всем деревьям.

Покажем на датасете Iris, как это считается шаг за шагом.

In [ ]:
iris = load_iris()
data_iris = pd.DataFrame(iris['data'], columns=iris['feature_names'])
target_iris = pd.Series(iris['target']).map({0: 0, 1: 0, 2: 1})  # Virginica vs rest

rfc_iris = RandomForestClassifier(n_estimators=3, max_depth=3, random_state=17)
rfc_iris.fit(data_iris, target_iris)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for i, (ax, est) in enumerate(zip(axes, rfc_iris.estimators_)):
    plot_tree(est, ax=ax, filled=True,
              feature_names=iris['feature_names'],
              class_names=['Other', 'Virginica'],
              node_ids=True, fontsize=8)
    ax.set_title(f'Дерево {i+1}')
plt.suptitle('3 дерева случайного леса на датасете Iris', y=1.01, fontsize=12)
plt.tight_layout()

In [ ]:
# Важность признаков из леса
importances = pd.Series(rfc_iris.feature_importances_, index=iris['feature_names'])

plt.figure(figsize=(7, 3))
importances.sort_values().plot(kind='barh', color='steelblue', edgecolor='white')
plt.xlabel('Важность (снижение примеси Джини)')
plt.title('Важность признаков в случайном лесе (Iris)')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()

`petal length` и `petal width` значительно важнее `sepal`-признаков. Это соответствует интуиции: по форме лепестка ирисы различаются сильнее.

### Permutation Importance (важность через перестановки)

Альтернативный подход: перемешать значения признака $X_j$ (случайная перестановка) и посмотреть, насколько упала точность на OOB-выборке.

- Если признак важен → точность сильно упадёт.
- Если признак бесполезен → точность почти не изменится.

Этот метод более надёжен для коррелированных признаков.

### Практический пример: факторы рейтинга хостелов

Данные: средние оценки посетителей хостелов по различным параметрам. Целевая переменная — итоговый рейтинг на сайте.

In [ ]:
import os

hostel_path = "../../data/hostel_factors.csv"
if os.path.exists(hostel_path):
    hostel_data = pd.read_csv(hostel_path)
    feature_names = {
        'f1':  'Персонал',
        'f2':  'Бронирование',
        'f3':  'Заезд/выезд',
        'f4':  'Состояние номера',
        'f5':  'Общая кухня',
        'f6':  'Общее пространство',
        'f7':  'Доп. услуги',
        'f8':  'Удобства',
        'f9':  'Цена/качество',
        'f10': 'Клиентоориентированность',
    }
    hostel_data = hostel_data.rename(columns=feature_names)
    target_col = hostel_data.columns[-1]  # итоговый рейтинг
    feat_cols  = [c for c in hostel_data.columns if c in feature_names.values()]

    rf_hostel = RandomForestRegressor(n_estimators=1000, max_features=10, random_state=0)
    rf_hostel.fit(hostel_data[feat_cols], hostel_data[target_col])

    imp = pd.Series(rf_hostel.feature_importances_, index=feat_cols).sort_values()
    plt.figure(figsize=(8, 4))
    imp.plot(kind='barh', color='steelblue', edgecolor='white')
    plt.xlabel('Важность признака')
    plt.title('Что влияет на итоговый рейтинг хостела?')
    plt.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
else:
    print("Файл hostel_factors.csv не найден — пропускаем этот пример.")
    print("Используем датасет телеком-оператора для иллюстрации важности признаков.")

    rf_feat = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    rf_feat.fit(X_ch, y_ch)
    imp = pd.Series(rf_feat.feature_importances_, index=X_ch.columns).sort_values()

    plt.figure(figsize=(8, 5))
    imp.plot(kind='barh', color='steelblue', edgecolor='white')
    plt.xlabel('Важность признака')
    plt.title('Важность признаков для предсказания оттока клиентов')
    plt.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()

Важность признаков — мощный инструмент не только для интерпретации, но и для **отбора признаков**: можно убрать признаки с низкой важностью и ускорить модель без потери качества.

---
## 8. Плюсы и минусы случайного леса

### Плюсы

- **Высокое качество** предсказаний — на большинстве задач превосходит линейные методы и отдельные деревья.
- **Устойчивость к выбросам** — благодаря бутстраповской выборке выбросы влияют лишь на часть деревьев.
- **Не требует масштабирования** — признаки не нужно нормировать.
- **Встроенная оценка важности признаков** — ценный инструмент интерпретации и отбора признаков.
- **OOB-оценка качества** — не нужна отдельная кросс-валидация.
- **Хорошо работает «из коробки»** — даже без настройки параметров результат часто приемлемый.
- **Параллелизуется** — `n_jobs=-1` ускоряет обучение на всех ядрах процессора.

### Минусы

- **Медленнее** одиночных деревьев — нужно обучить $N$ деревьев.
- **Много памяти** — каждое дерево хранится в памяти.
- **Хуже интерпретируется**, чем одиночное дерево — нельзя нарисовать весь лес.
- **Плохо работает на очень разреженных данных** — для текстов лучше подходит логистическая регрессия.
- **Не умеет экстраполировать** — за пределами диапазона обучающих данных предсказывает константу (как любое дерево).
- **Слабее бустинга** на большинстве задач в соревновательном ML — XGBoost, LightGBM часто точнее.

---
## 9. Практика

Для заданий используем датасет телеком-оператора.

In [ ]:
print(f"Объектов: {X_ch.shape[0]}, признаков: {X_ch.shape[1]}")
print(f"Доля оттока: {y_ch.mean():.2%}")
X_ch.head(3)

### Задание 1
Обучите `RandomForestClassifier` с параметрами по умолчанию и измерьте точность (accuracy) на 5-fold кросс-валидации.

Затем включите `oob_score=True` и сравните OOB-точность с CV-точностью. Насколько они близки?

In [ ]:
# Ваш код здесь

### Задание 2
Постройте кривую зависимости качества (ROC-AUC, CV=5) от числа деревьев `n_estimators` в диапазоне `[5, 10, 20, 30, 50, 75, 100, 150, 200]`.

Начиная с какого числа деревьев качество выходит на плато?

In [ ]:
# Ваш код здесь

### Задание 3
С помощью `GridSearchCV` (метрика `roc_auc`, `cv=5`) подберите оптимальные параметры для случайного леса:

```python
params = {
    'max_depth':         [5, 10, 15, None],
    'min_samples_leaf':  [1, 3, 5],
    'max_features':      ['sqrt', 0.5, 0.8],
}
```

Выведите лучшие параметры и итоговый ROC-AUC.

In [ ]:
# Ваш код здесь

### Задание 4
Визуализируйте важность признаков лучшей модели из задания 3.

Какие 3 признака наиболее важны для предсказания оттока? Совпадает ли это с находками из визуального анализа (Тема 2)?

In [ ]:
# Ваш код здесь

### Задание 5 (повышенная сложность)
Сравните `RandomForestClassifier` и `ExtraTreesClassifier` на данных телеком-оператора:

1. Подберите оптимальные параметры для каждого (можно использовать те же сетки параметров).
2. Сравните ROC-AUC на кросс-валидации.
3. Сравните важность признаков у двух моделей — есть ли различия?

In [ ]:
# Ваш код здесь

---
## Полезные ресурсы

- [Документация sklearn: RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
- [Документация sklearn: ExtraTreesClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.ExtraTreesClassifier.html)
- [BaggingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html)
- Статья Бреймана [«Random Forests»](https://link.springer.com/article/10.1023/A:1010933404324) (2001) — оригинальная работа
- [«Understanding Random Forests»](https://arxiv.org/abs/1407.7502) — подробный разбор теории